# Chapter 8 — You Cannot Optimize What You Cannot Measure

**Book alignment:** DSPy From First Principles, Chapter 8

**Question this notebook isolates:** Does a frozen program scored under a frozen protocol produce a reproducible baseline whose aggregate hides the per-case diagnosis?

In [ ]:
from pathlib import Path
import random
import sys

random.seed(13)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy  # imported for construction only; no LM is ever called

from common.data import canonical_split, dataset_fingerprint, split_fingerprint
from common.dspy_program import EditorialRewriteProgram
from common.fingerprints import fingerprint
from common.metrics import editorial_metric, metric_identity_record

## Freeze the baseline identity

A baseline is a program identity, a model identity, a dataset identity, a metric identity, and a protocol. Reconstructing the program twice must give the same fingerprint, and the accepted program must carry zero demonstrations.

In [ ]:
split = canonical_split()
program_a = EditorialRewriteProgram()
program_b = EditorialRewriteProgram()

metric_identity = metric_identity_record()
baseline_identity = {
    "program_id": "EditorialRewriteProgram",
    "module_strategy": "analyze_predict__rewrite_predict__assess_predict",
    "program_fingerprint": fingerprint(program_a.dump_state()),
    "dataset_fingerprint": dataset_fingerprint(),
    "split_fingerprint": split_fingerprint(split),
    "split_role": "dev",
    "metric_name": "editorial_metric",
    "metric_version": "v1",
    "optimization_allowed": False,
}

({
    "program_fingerprint": baseline_identity["program_fingerprint"][:16],
    "train": len(split.train),
    "dev": len(split.dev),
    "holdout": len(split.holdout),
    "metric_versions": metric_identity["versions"],
})

In [ ]:
assert baseline_identity["program_fingerprint"] == fingerprint(program_b.dump_state())
assert all(len(demos) == 0 for _, predictor in program_a.named_predictors() for demos in [predictor.demos])
assert (len(split.train), len(split.dev), len(split.holdout)) == (26, 11, 7)
assert set(split.dev_ids).isdisjoint(split.holdout_ids)
assert set(split.train_ids).isdisjoint(split.dev_ids)

print("baseline identity stable:", baseline_identity["program_fingerprint"][:12])
print("dev cases:", ", ".join(split.dev_ids))

## The aggregate hides the diagnosis

Score the deterministic no-edit baseline on every development case under frozen v1, then group by the expected failure mode. The mean is the least informative number computed here.

In [ ]:
rows = []
for case in split.dev:
    breakdown = editorial_metric(case, case.sentence, version="v1")
    rows.append({
        "case_id": case.case_id,
        "target_failure": case.target_failure,
        "score": breakdown.score,
        "changed": breakdown.changed_score,
        "scope": breakdown.scope_length_score,
        "overlap": breakdown.reference_overlap_score,
    })

mean_all = sum(r["score"] for r in rows) / len(rows)
by_failure = {}
for r in rows:
    by_failure.setdefault(r["target_failure"], []).append((r["case_id"], round(r["score"], 3)))

({"no_edit_dev_mean": round(mean_all, 4)}, {k: v for k, v in sorted(by_failure.items())})

In [ ]:
identity_cases = {r["case_id"]: r for r in rows if r["case_id"] in ("ed-034", "ed-037")}
assert set(identity_cases) == {"ed-034", "ed-037"}
for case_id, r in identity_cases.items():
    # reference equals source: full scope and overlap, zero change credit
    assert (r["changed"], r["scope"], r["overlap"]) == (0.0, 1.0, 1.0), case_id
    assert abs(r["score"] - 0.65) < 1e-9, case_id  # 0.35*0 + 0.25*1 + 0.40*1

floor_rows = [r for r in rows if abs(r["score"] - 0.65) < 1e-9]
assert len(floor_rows) >= 2

print(f"no-edit dev mean: {mean_all:.4f}")
print(f"cases pinned at the 0.65 floor: {sorted(r['case_id'] for r in floor_rows)}")
print("correct restraint forfeits the 0.35 changed component by construction")

## What we earned

A frozen program, frozen cases, and frozen v1 produce a reproducible baseline identity — and the per-case table already exposes a defect: where no edit is the right answer, v1 caps correct restraint at 0.65 because it pays for change regardless of need.

Notebook 09 / Chapter 9 stops using the metric and starts attacking it: what else does this objective reward that it should not?